In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. 데이터 파일 읽기
df = pd.read_csv('/workspaces/Daegun26Veritas/data/20723data.csv')

# 2. 컬럼명 예외 처리 (KeyError 방지)
# 'content'와 'class' 컬럼이 정확히 존재하면 사용하고, 깨져있다면 마지막 두 컬럼을 강제로 매핑합니다.
if 'content' in df.columns and 'class' in df.columns:
    df = df[['content', 'class']]
else:
    df = df.iloc[:, [-2, -1]]
    df.columns = ['content', 'class']

# 3. 결측치(빈 칸) 데이터가 있으면 행 전체 삭제 후 인덱스 재정렬
df = df.dropna().reset_index(drop=True)

# 4. 문자열 또는 비연속적인 클래스(라벨)를 0부터 시작하는 연속된 정수로 변환
df['label'] = df['class'].astype('category').cat.codes
num_labels = df['label'].nunique()

print(f"✅ 데이터 전처리 완료! 총 샘플 수: {len(df)}, 클래스 종류 수: {num_labels}")

# 5. 학습 데이터와 검증 데이터 분할 (80% 학습, 20% 검증)
# stratify 옵션을 주어 학습/검증셋에 스팸과 정상 문자의 비율이 고르게 들어가도록 합니다.
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['content'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label'].tolist()
)

✅ 데이터 전처리 완료! 총 샘플 수: 19009, 클래스 종류 수: 3


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. 데이터 파일 읽기
df = pd.read_csv('/workspaces/Daegun26Veritas/data/20723data.csv')

# 2. 컬럼명 예외 처리 (KeyError 방지)
# 'content'와 'class' 컬럼이 정확히 존재하면 사용하고, 깨져있다면 마지막 두 컬럼을 강제로 매핑합니다.
if 'content' in df.columns and 'class' in df.columns:
    df = df[['content', 'class']]
else:
    df = df.iloc[:, [-2, -1]]
    df.columns = ['content', 'class']


# 3. 결측치(빈 칸) 데이터가 있으면 행 전체 삭제 후 인덱스 재정렬
df = df.dropna().reset_index(drop=True)

# 4. 문자열 또는 비연속적인 클래스(라벨)를 0부터 시작하는 연속된 정수로 변환
df['label'] = df['class'].astype('category').cat.codes
num_labels = df['label'].nunique()

print(f"✅ 데이터 전처리 완료! 총 샘플 수: {len(df)}, 클래스 종류 수: {num_labels}")

# 5. 학습 데이터와 검증 데이터 분할 (80% 학습, 20% 검증)
# stratify 옵션을 주어 학습/검증셋에 스팸과 정상 문자의 비율이 고르게 들어가도록 합니다.
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['content'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label'].tolist()
)

✅ 데이터 전처리 완료! 총 샘플 수: 19009, 클래스 종류 수: 3


In [5]:
!pip install torch transformers scikit-learn
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

# 1. 한국어 전용 사전학습 BERT 모델의 토크나이저 로드
MODEL_NAME = "kykim/bert-kor-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2. PyTorch용 Custom Dataset 클래스 정의
class SpamDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        # 텍스트들을 일괄 토큰화 (길이는 최대 128 토큰으로 제한 및 패딩 처리)
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors=None
        )
        self.labels = labels

    def __getitem__(self, idx):
        # 각 인덱스(idx)에 해당하는 토큰 데이터들을 텐서(Tensor) 구조로 변환
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

# 3. 학습용 및 검증용 데이터셋 인스턴스 생성
train_dataset = SpamDataset(train_texts, train_labels, tokenizer)
val_dataset = SpamDataset(val_texts, val_labels, tokenizer)

print("✅ 토큰화 및 데이터셋 객체 생성 완료!")

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 877.3 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 24.0 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 36.7 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 40.4 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 7.2 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 2.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.6/197.6 MB 2.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 2.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

/home/vscode/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 토큰화 및 데이터셋 객체 생성 완료!


In [6]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

# 1. 분류용 BERT 모델 로드 (출력 라벨 개수 설정)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

# 2. 학습 하이퍼파라미터 및 옵션 설정
training_args = TrainingArguments(
    output_dir='./results',          # 학습 결과(체크포인트)가 저장될 폴더
    num_train_epochs=3,              # 전체 데이터를 몇 번 반복해서 학습할 것인지
    per_device_train_batch_size=16,  # 한 번에 모델이 학습할 데이터 개수
    per_device_eval_batch_size=16,   # 평가 시 사용할 배치 사이즈
    warmup_steps=300,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",           # 매 에포크(Epoch) 끝날 때마다 검증 진행
    save_strategy="epoch",
    load_best_model_at_end=True,     # 학습 종료 후 가장 성적이 좋은 최적의 모델을 자동 로드
    metric_for_best_model="f1",      # 불균형 데이터이므로 f1-score를 기준으로 최고 모델 판단
    fp16=torch.cuda.is_available(),  # GPU가 연산을 지원할 경우 16비트 연산으로 속도 향상
    report_to="none"                 # 불필요한 외부 툴(wandb 등) 연동 비활성화
)

# 3. 평가지표 계산 함수 정의
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'f1': f1}

# 4. 트레이너(Trainer) 객체 생성 및 학습 실행
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("🚀 모델 학습을 시작합니다...")
trainer.train()
print("✅ 모델 학습 완료 및 베스트 모델 저장 완료!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10361.83it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: kykim/bert-kor-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architectu

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

In [3]:
import torch
import torch.nn.functional as F

def predict_spam_with_label_name(text):
    """
    입력된 한 줄의 텍스트를 AI 모델로 분류하고,
    숫자 대신 "스팸" 혹은 "정상"이라는 명확한 한글 결과와 확률을 반환합니다.
    """
    # 1. 입력 텍스트 토큰화
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)

    # 2. GPU/CPU 장치 지정 및 데이터 이동
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 3. 모델 추론(예측) 진행
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    # 4. 확률값 계산
    logits = outputs.logits
    probs = F.softmax(logits, dim=-1)

    # 5. 가장 높은 확률을 가진 클래스 번호(숫자)와 신뢰도 추출
    pred_class_idx = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][pred_class_idx].item()

    # =================================================================
    # 🌟 [핵심 수정 부분] 숫자를 읽기 쉬운 텍스트로 변환하는 매핑 사전
    # =================================================================
    # 본인의 데이터셋 class 종류에 맞게 아래 번호와 문구를 수정하셔도 됩니다.
    class_map = {
        0: "정상 메시지",
        1: "광고성 스팸",
        2: "보이스피싱/스미싱 위험",
        3: "정상 알림톡(쿠팡 등)"
    }

    # 만약 사전에 없는 번호가 나올 경우를 대비한 예외 처리
    result_label = class_map.get(pred_class_idx, f"알 수 없는 클래스({pred_class_idx})")

    # 숫자가 아닌 글자 형태의 결과(result_label)와 확률을 반환합니다.
    return result_label, confidence

In [4]:
print("🔮 스팸/피싱 문자 판별 시스템이 준비되었습니다.")

# 1. 사용자로부터 직접 검사할 문장 입력받기
user_input = input("검사할 문자 내용을 입력하고 Enter를 누르세요: ")

# 2. 수정된 함수 호출 (결과가 숫자가 아닌 '글자'로 나옴)
status_text, score = predict_spam_with_label_name(user_input)

# 3. 결과 화면 출력
print("\n" + "="*50)
print(f"💬 입력한 문자: {user_input}")
print(f"🤖 AI 분석 결과: [ {status_text} ] 문장일 확률이 높습니다.")


🔮 스팸/피싱 문자 판별 시스템이 준비되었습니다.


NameError: name 'tokenizer' is not defined